# Binned cluster number counts $N(q, z)$: $Y_{500c}$ vs $Y_{5R500c}$

Side-by-side comparison of the FLAMINGO L2p8_m9 **mock catalogues** (two Y apertures)
against a single **cosmocnc_jax** theory prediction per hydrostatic bias **B** ($B=1$ and
$B=1.35$, i.e. $B=1/(1-b)$).

| B | Aperture | Mock catalogue | $N(q>5)$ |
|---|----------|----------------|----------|
| 1.00 | $Y_{5R500c}$ | `halo_catalogue_..._arnaudB1.csv` | 2392 |
| 1.00 | $Y_{500c}$ | `halo_catalogue_..._arnaudB1_Y500c.csv` | 2023 |
| 1.35 | $Y_{5R500c}$ | `halo_catalogue_..._arnaudB135.csv` | 885 |
| 1.35 | $Y_{500c}$ | `halo_catalogue_..._arnaudB135_Y500c.csv` | 768 |

All use the same CNC bin grid (10 redshift $\times$ 5 SNR bins, $5<q<40$).

**Theory** (`cosmocnc_jax`, hmfast HMF): one $N(q,z)$ from the GNFW central Compton-$y$
$y_0(M,z)$ and Planck-style selection $q=y_0/\sigma_{y_0}(\theta_{500})$ (`q_planck_sim`,
SZiFi `immf6` noise). Fiducials: $\Omega_m=0.306$, $\sigma_8=0.808$, $\alpha=1.12$,
$\sigma_{\ln Y}=0.173$, $A_{SZ}=-4.3054$, `bias_sz` $=1/B$. The same $y_0$ theory applies
to both mock apertures; they differ only in how SOAP integrates $Y$.


In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "text.usetex": False,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Helvetica", "Arial", "sans-serif"],
    "mathtext.fontset": "dejavusans",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
})

import pandas as pd

from flamingo import paths

CNC = paths.ROOT / "data" / "cnc"

CASES = {
    1.00: {
        "y5r500_npz": CNC / "observed_mock_cnc_arnaudB1.npz",
        "y500_npz": CNC / "observed_cnc_binned_qgt5_arnaudB1_Y500c.npz",
        "y5r500_cat": paths.HYDRO / "catalogue" / "halo_catalogue_M500c_5e13_zlt3_y0q_arnaudB1.csv",
    },
    1.35: {
        "y5r500_npz": CNC / "observed_cnc_binned_qgt5_arnaudB135.npz",
        "y500_npz": CNC / "observed_cnc_binned_qgt5_arnaudB135_Y500c.npz",
    },
}

data = {}
for B, paths_npz in CASES.items():
    d5 = np.load(paths_npz["y5r500_npz"], allow_pickle=True)
    d1 = np.load(paths_npz["y500_npz"], allow_pickle=True)
    if "n_total_qgt5" in d5.files:
        n_qgt5_5R500 = int(d5["n_total_qgt5"])
    else:
        q = pd.read_csv(paths_npz["y5r500_cat"], usecols=["q"], comment="#")["q"].to_numpy(float)
        n_qgt5_5R500 = int(np.sum(q > 5.0))
    data[B] = {
        "N2d_5R500": d5["N2d"],
        "N2d_500": d1["N2d"],
        "n_qgt5_5R500": n_qgt5_5R500,
        "n_qgt5_500": int(d1["n_total_qgt5"]),
    }

z_edges = d5["z_edges"]
q_edges = d5["q_edges"]
z_centres = d5["z_centres"]
q_centres = d5["q_centres"]

for B in sorted(data):
    d = data[B]
    print(
        f"B={B:.2f}  Y_5R500c N(q>5)={d['n_qgt5_5R500']:4d}  "
        f"(grid {int(d['N2d_5R500'].sum()):4d})  |  "
        f"Y_500c N(q>5)={d['n_qgt5_500']:4d}  "
        f"(grid {int(d['N2d_500'].sum()):4d})"
    )
print(f"grid: {data[1.00]['N2d_5R500'].shape[0]} z bins x {data[1.00]['N2d_5R500'].shape[1]} q bins")

B=1.00  Y_5R500c N(q>5)=2392  (grid 2376)  |  Y_500c N(q>5)=2023  (grid 2007)
B=1.35  Y_5R500c N(q>5)= 885  (grid  884)  |  Y_500c N(q>5)= 768  (grid  767)
grid: 10 z bins x 5 q bins


In [2]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

def plot_n2d_annotated(ax, N2d, z_edges, q_edges, z_centres, q_centres, title, *, vmax=None, as_int=True):
    """Heatmap of N(z,q) with counts annotated on non-empty bins."""
    arr = np.asarray(N2d, float)
    data = arr.T
    masked = np.ma.masked_where(data <= 0, data)
    if vmax is None:
        vmax = max(float(data.max()), 1.0)
    im = ax.pcolormesh(
        z_edges,
        np.arange(len(q_edges)),
        masked,
        cmap="viridis",
        shading="flat",
        vmin=0,
        vmax=vmax,
    )
    ax.set_yticks(np.arange(len(q_centres)) + 0.5)
    ax.set_yticklabels([f"{q_edges[j]:.0f}" for j in range(len(q_centres))], fontsize=8)
    ax.set_xlabel(r"redshift $z$")
    ax.set_ylabel(r"SNR $q$ (lower edge)")
    ax.set_title(title)
    for i in range(len(z_centres)):
        for j in range(len(q_centres)):
            v = float(arr[i, j])
            if v > 0:
                txt = f"{int(v)}" if as_int else f"{v:.0f}"
                if txt == "0":
                    continue
                ax.text(
                    z_centres[i],
                    j + 0.5,
                    txt,
                    ha="center",
                    va="center",
                    fontsize=7,
                    color="w",
                )
    return im


def plot_y500_vs_y5r500(B, d, *, vmax=None, tag="mock catalogue", as_int=True):
    N2d_5R500 = d["N2d_5R500"]
    N2d_500 = d["N2d_500"]
    n_qgt5_5R500 = d["n_qgt5_5R500"]
    n_qgt5_500 = d["n_qgt5_500"]

    if vmax is None:
        vmax = max(float(np.max(N2d_5R500)), float(np.max(N2d_500)), 1.0)

    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.0), sharey=True)

    plot_n2d_annotated(
        axes[0], N2d_5R500, z_edges, q_edges, z_centres, q_centres,
        rf"$Y_{{5R500c}}$  ($N_{{q>5}}={n_qgt5_5R500}$)", vmax=vmax, as_int=as_int,
    )
    im1 = plot_n2d_annotated(
        axes[1], N2d_500, z_edges, q_edges, z_centres, q_centres,
        rf"$Y_{{500c}}$  ($N_{{q>5}}={n_qgt5_500}$)", vmax=vmax, as_int=as_int,
    )

    divider = make_axes_locatable(axes[1])
    cax = divider.append_axes("right", size="4%", pad=0.55)
    cb = fig.colorbar(im1, cax=cax)
    cb.set_label("clusters per bin")

    fig.suptitle(
        rf"FLAMINGO L2p8_m9 CNC bins (B={B:.2f}, A10 GNFW, {tag}): binned $N(q,z)$, $5<q<40$",
        fontsize=12,
        y=0.98,
    )
    fig.subplots_adjust(top=0.86, wspace=0.35)
    return fig


def plot_theory_y0(B, N2d, n_total, *, vmax=None, as_int=False):
    """Single-panel N(q,z) from cosmocnc_jax y0 theory."""
    if vmax is None:
        vmax = max(float(np.max(N2d)), 1.0)

    fig, ax = plt.subplots(figsize=(7.0, 5.0))
    im = plot_n2d_annotated(
        ax, N2d, z_edges, q_edges, z_centres, q_centres,
        rf"$y_0$ theory  ($N_{{q>5}}={n_total:.0f}$)", vmax=vmax, as_int=as_int,
    )
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4%", pad=0.55)
    cb = fig.colorbar(im, cax=cax)
    cb.set_label("clusters per bin")
    fig.suptitle(
        rf"cosmocnc_jax $N(q,z)$ from GNFW $y_0$ (B={B:.2f}, fiducial $A_{{SZ}}=-4.3054$)",
        fontsize=12,
        y=0.98,
    )
    fig.subplots_adjust(top=0.86)
    return fig


vmax_obs = max(
    max(float(data[B]["N2d_5R500"].max()), float(data[B]["N2d_500"].max()))
    for B in data
)

for B in sorted(data):
    plot_y500_vs_y5r500(B, data[B], vmax=vmax_obs, tag="mock catalogue", as_int=True)
    plt.show()


## Theory: cosmocnc_jax (GNFW $y_0$ + fiducials)

One expected $N(q,z)$ per $B$ from `cosmocnc_jax.cluster_number_counts` with
`q_planck_sim` ($q=y_0/\sigma_{y_0}$), hmfast Tinker08 HMF, and `bias_sz = 1/B`.
Compare to **both** mock panels above: theory does not split by $Y_{500c}$ vs $Y_{5R500c}$.


In [3]:
from pathlib import Path

import os

# cosmocnc_jax / TF init (same pattern as cobaya/likelihood/cnc_likelihood.py)
os.environ.setdefault("JAX_ENABLE_X64", "1")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "true")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.30")
_cuda = os.environ.get("CUDA_VISIBLE_DEVICES", "0")
os.environ["CUDA_VISIBLE_DEVICES"] = ""
import tensorflow as tf
try:
    tf.config.set_visible_devices([], "GPU")
except Exception:
    pass
os.environ["CUDA_VISIBLE_DEVICES"] = _cuda

from cosmocnc_jax import cluster_number_counts
from cosmocnc_jax.params import (
    cnc_params_default,
    cosmo_params_default,
    scaling_relation_params_default,
)

COSMOCNC_PKG = Path("/scratch/scratch-lxu/agent_dev/auto_research_agent/cosmocnc_jax/cosmocnc_jax")

# FLAMINGO / D3A fiducials (nb27)
OM_TRUTH, S8_TRUTH = 0.306, 0.808
H_FID, OMB_FID = 0.681, 0.022539
OMEGA_CDM = OM_TRUTH * H_FID**2 - OMB_FID - 0.000644
FID_COSMO = {
    "h": H_FID,
    "Ob0h2": OMB_FID,
    "Oc0h2": OMEGA_CDM,
    "n_s": 0.967,
    "tau_reio": 0.0544,
    "m_nu": 0.06,
    "sigma_8": S8_TRUTH,
}
FID_SCAL = {
    "A_szifi": -4.3054,
    "alpha_szifi": 1.12,
    "sigma_lnq_szifi": 0.173,
}

_cnc_params = dict(cnc_params_default)
_cnc_params.update(
    {
        "survey_sr": str(COSMOCNC_PKG / "surveys/survey_sr_planck_sim.py"),
        "survey_cat": str(COSMOCNC_PKG / "surveys/survey_cat_planck_sim.py"),
        "tszsbi_noise_dir": "/scratch/scratch-lxu/tszsbi/noise_files",
        "tszsbi_filter_name": "immf6",
        "load_catalogue": False,
        "likelihood_type": "binned",
        "binned_lik_type": "z_and_obs_select",
        "data_lik_from_abundance": False,
        "obs_select": "q_planck_sim",
        "observables": [["q_planck_sim"]],
        "obs_select_min": float(q_edges[0]),
        "obs_select_max": float(q_edges[-1]),
        "z_min": float(z_edges[0]),
        "z_max": float(z_edges[-1]),
        "bins_edges_z": z_edges,
        "bins_edges_obs_select": q_edges,
        "n_points": 2048,
        "n_z": 50,
        "M_min": 5.0e13,
        "M_max": 1.0e16,
        "planck_sim_M_pivot": 2.1e14,
        "cosmology_tool": "hmfast",
        "hmfast_path": "/scratch/scratch-lxu/agent_dev/auto_research_agent/hmfast/src",
        "hmf_calc": "cnc",
        "cosmo_param_density": "physical",
        "cosmo_amplitude_parameter": "sigma_8",
        "cosmocnc_verbose": "none",
    }
)

_cnc = cluster_number_counts(cnc_params=_cnc_params)
_cosmo = dict(cosmo_params_default)
_cosmo.update(FID_COSMO)
_cnc.cosmo_params = _cosmo
_cnc.scal_rel_params = dict(scaling_relation_params_default)
_cnc.initialise()
print("cosmocnc_jax initialised")


def theory_n2d(B):
    """Single y0-based N(q,z) for hydrostatic bias B."""
    scal = dict(scaling_relation_params_default)
    scal.update(FID_SCAL)
    scal["bias_sz"] = 1.0 / B
    _cnc.update_params(_cosmo, scal)
    _cnc.get_log_lik_binned()
    return np.asarray(_cnc.n_binned, float)


theory = {}
for B in sorted(data):
    N = theory_n2d(B)
    theory[B] = {"N2d": N, "n_qgt5": float(N.sum())}
    print(f"theory B={B:.2f}  y0  N={N.sum():.1f}")

vmax_th = max(float(theory[B]["N2d"].max()) for B in theory)

for B in sorted(theory):
    plot_theory_y0(B, theory[B]["N2d"], theory[B]["n_qgt5"], vmax=vmax_th, as_int=False)
    plt.show()


I0000 00:00:1782461855.061671 2751372 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782461855.062048 2751372 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1782461855.088510 2751372 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1782461856.159308 2751372 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782461856.159687 2751372 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1782461856.290791 2751372 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
I0000 00:00:1782461856.290825 2751372 cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES=""
I0000 00:00:1782461856.290835 2751372 cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to an empty string - this hides all GPUs from CUDA
I0000 00:00:1782461856.290844 2751372 cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get mor

defaulting to  /rds-d4/user/iz221/hpc-work/cosmopower/
defaulting to  /scratch/scratch-lxu/agent_dev/auto_research_agent/cosmocnc_jax/cosmocnc_jax/


cosmo params {'Om0': 0.3046113536929582, 'Ob0': 0.048600464463376604, 'Ob0h2': 0.022539, 'Oc0h2': 0.11872786600000002, 'h': 0.681, 'A_s': np.float64(2.1054062513502893e-09), 'n_s': 0.967, 'm_nu': 0.06, 'sigma_8': 0.8079999999999998, 'tau_reio': 0.0544, 'w0': -1.0, 'Onu0': 0.0013890593206797887, 'N_eff': 3.046, 'k_cutoff': 100000000.0, 'ps_cutoff': 1}


cosmocnc_jax initialised


theory B=1.00  y0  N=3653.9
theory B=1.35  y0  N=1459.6


## Tabulated counts

Rows = redshift bins (low to high $z$); columns = SNR bins (low to high $q$).


In [4]:
q_hdr = "  ".join(f"q={q_centres[j]:5.1f}" for j in range(len(q_centres)))

for B in sorted(data):
    for label, key in [("Y_5R500c", "N2d_5R500"), ("Y_500c", "N2d_500")]:
        N2d = data[B][key]
        print(f"\n=== mock B={B:.2f}  {label} ===")
        print(q_hdr)
        for i in range(len(z_centres)):
            row = "  ".join(f"{int(N2d[i, j]):5d}" for j in range(len(q_centres)))
            print(f"z={z_centres[i]:.3f}  {row}")

for B in sorted(theory):
    N2d = theory[B]["N2d"]
    print(f"\n=== theory B={B:.2f}  y0 ===")
    print(q_hdr)
    for i in range(len(z_centres)):
        row = "  ".join(f"{N2d[i, j]:8.1f}" for j in range(len(q_centres)))
        print(f"z={z_centres[i]:.3f}  {row}")



=== mock B=1.00  Y_5R500c ===
q=  6.2  q=  9.3  q= 14.1  q= 21.4  q= 32.5
z=0.055    291    158     94     52     16
z=0.154    402    181     74     24     12
z=0.254    308    119     43     14      2
z=0.353    185     56     24      7      0
z=0.453    101     38     11      1      0
z=0.552     54     21      4      0      0
z=0.652     43     11      1      0      0
z=0.751     25      0      0      0      0
z=0.851      2      0      0      0      0
z=0.950      2      0      0      0      0

=== mock B=1.00  Y_500c ===
q=  6.2  q=  9.3  q= 14.1  q= 21.4  q= 32.5
z=0.055    187    127     77     37      7
z=0.154    299    146     59     22     12
z=0.254    252    103     41     10      2
z=0.353    168     61     22      7      1
z=0.453    104     39     13      5      0
z=0.552     65     24      7      0      0
z=0.652     43     15      2      0      0
z=0.751     19      9      0      0      0
z=0.851     12      0      0      0      0
z=0.950     10      0      0      0